#Autonomous Driving Environment

## Environment

In [ ]:
#environment.py

import random
import sys

MAX_VIEW = 10 # DO NOT CHANGE THIS
INIT_LOCATION_WEIGHT = 1/5 # DO NOT CHANGE THIS
NON_FORWARD_WEIGHT = 3 # DO NOT CHANGE THIS

class Environment:
    def __init__(self, height=10, width=5, occupancy=0.1, init=0.2, seed=5):
        self.height = height
        self.width = width
        self.occupancy = occupancy
        self.init = init
        self.cars = {}
        self.road = [[0 for _ in range(width)] for _ in range(height)]
        self.crash = None
        self.done = None

        self.actions = ['F', 'L', 'R', 'W']
        self.score = 0
        self.__generate_environment(seed)

    def __generate_environment(self, seed):
        """
        Description
        -----------
        This function is used to populate self.road and self.cars
        self.cars is a dict that holds the location of all cars on the road including the autonomous agent's (idx 0)
        self.road is a 2D list (grid) representing the discretized road environment with the following markers:
            0: empty cell
            1: obstacle car
            2: autonomous agent (AA)
           -1: crash

        The convention for positions, like an array, is that (r,c) = (0,0) is the lower left corner, r increases
        vertically and c increases horizontally.

        """
        random.seed(seed)

        # Number of obstacles should not exceed OCCUPANCY cells in the environment
        n_obstacles = int(self.height * self.width * self.occupancy)

        coordinate_list = [(r, c) for r in range(self.height) for c in range(self.width)]
        obstacle_list = random.sample(coordinate_list, n_obstacles)

        # Mark the AA
        r = int(self.height * self.init)
        c = int(self.width * self.init)
        self.road[r][c] = 2
        self.cars[0] = (r, c) #AA location is stored in the 0th index of cars dict

        # Mark obstacle cars
        idx = 1
        for r, c in obstacle_list:
            if (r, c) != self.cars[0]:
                self.road[r][c] = 1
                self.cars[idx] = (r, c)
                idx += 1

    def get_next_cell(self, idx, action):
        """
        Description
        -----------
        This function finds a car's next location given a car's index and action it may perform.
        If the action performed leads a car outside the road, its next location is updated to None

        """
        loc = self.cars[idx]

        if action == 'F':
            if loc[0]+1 < self.height:
                return (loc[0]+1, loc[1])
            elif loc[0]+1 == self.height: # moving forward leads out of the road
                return None
        elif action == 'L':
            return (loc[0], loc[1]-1)
        elif action == 'R':
            return (loc[0], loc[1]+1)
        elif action == 'W':
            return loc

    def get_possible_next_cells(self, idx):
        """
        Description
        -----------
        This function finds all possible next locations given a car's index (except AA's).

        """
        next_cells = []
        for action in self.get_legal_actions(idx):
            next_loc = self.get_next_cell(idx, action)
            next_cells.append(next_loc)

        return next_cells

    def get_legal_actions(self, idx):
        """
        Description
        -----------
        This function finds all possible legal actions given a car's index by considering its current location.
        Actions leading to locations where a car (except AA) is already present is considered illegal for other cars.
        Wait action is considered legal by default for other cars.

        """
        loc = self.cars[idx]
        legal_actions = []
        # Check all three directions a car can move to
        if loc[0]+1 < self.height and self.road[loc[0]+1][loc[1]] != 1: # Forward action
            legal_actions.append('F')
        elif loc[0]+1 == self.height: # moving forward leads out of the road
            legal_actions.append('F')

        if loc[1]-1 >= 0 and self.road[loc[0]][loc[1]-1] != 1: # Left action
            legal_actions.append('L')

        if loc[1]+1 < self.width and self.road[loc[0]][loc[1]+1] != 1: # Right action
            legal_actions.append('R')

        legal_actions.append('W') # Wait action

        return legal_actions

    def get_weights_of_actions(self, actions):
        n = len(actions)
        weights = []

        for a in actions:
            if a == 'F':
                weights.append(10 - NON_FORWARD_WEIGHT) #Forward action has the highest weight
                n -= 1
            else:
                weights.append(NON_FORWARD_WEIGHT/n)

        return weights

    def __update_environment(self):
        """
        Description
        -----------
        This function updates the position of all cars (excluding AA's) in the environment everytime the step function is called.
        Each car changes lane or waits with weight NON_FORWARD_WEIGHT.
        Cars that move forward beyond the scope of the road are no longer considered part of the environment and their location updates to None.

        """

        for idx, loc in self.cars.items():
            if idx == 0 or loc == None:
                continue
            legal_actions = self.get_legal_actions(idx)
            action_weights = self.get_weights_of_actions(legal_actions)
            action = random.choices(legal_actions, weights=action_weights, k=1)[0]
            self.perform_car_action(idx, action)

    def perform_car_action(self, idx, action):
        """
        Description
        -----------
        This function updates the position of a car in self.cars (except AA) as a result of the action performed in the environment.

        """
        loc = self.cars[idx]
        next_cell = self.get_next_cell(idx, action)
        if next_cell == None: # car has crossed the road
            self.road[loc[0]][loc[1]] = 0 # Erase current marker
        else:
            self.road[loc[0]][loc[1]] = 0 # Erase current marker
            self.road[next_cell[0]][next_cell[1]] = 1 # Add new marker

        self.cars[idx] = next_cell # update location of car in self.cars

    def perform_AA_action(self, action):
        """
        Description
        -----------
        This function updates the position of the AA's as a result of the action performed in the environment everytime the step function is called.

        """
        loc = self.cars[0]
        if action == 'F':
            if loc[0]+1 < self.height: # Next cell is within the scope of the road
                if self.road[loc[0]+1][loc[1]] == 1: # 'A' is crashing into a car
                    self.crash = True
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]+1][loc[1]] = -1 # Add marker for crash
                    self.cars[0] = (-1, -1)
                else:
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]+1][loc[1]] = 2 # Add new marker
                    self.cars[0] = (loc[0]+1, loc[1])
                    self.score += 1
            else: # 'A' has successfully navigated till the end of the road
                self.done = True
                self.cars[0] = None
        elif action == 'L':
            if loc[1]-1 >= 0: # Next cell is within the scope of the road
                if self.road[loc[0]][loc[1]-1] == 1: # A is crashing into a car
                    self.crash = True
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]][loc[1]-1] = -1 # Add marker for crash
                    self.cars[0] == (-1, -1)
                else:
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]][loc[1]-1] = 2 # Add new marker
                    self.cars[0] = (loc[0], loc[1]-1)
            else: # 'A' is moving out of the scope of the road (i.e. driving beyond the left shoulder of the road)
                self.crash = True
                if self.road[loc[0]][loc[1]] != 1:
                    self.road[loc[0]][loc[1]] = 0 # Erase current marker
                self.cars[0] == (-1, -1)
        elif action == 'R':
            if loc[1]+1 < self.width: # Next cell is within the scope of the road
                if self.road[loc[0]][loc[1]+1] == 1: # A is crashing into a car
                    self.crash = True
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]][loc[1]+1] = -1 # Add marker for crash
                    self.cars[0] == (-1, -1)
                else:
                    if self.road[loc[0]][loc[1]] != 1:
                        self.road[loc[0]][loc[1]] = 0 # Erase current marker
                    self.road[loc[0]][loc[1]+1] = 2 # Add new marker
                    self.cars[0] = (loc[0], loc[1]+1)
            else: # 'A' is moving out of the scope of the road (i.e. driving beyond the right shoulder of the road)
                self.crash = True
                if self.road[loc[0]][loc[1]] != 1:
                    self.road[loc[0]][loc[1]] = 0 # Erase current marker
                self.cars[0] == (-1, -1)
        elif action == 'W':
            if self.road[loc[0]][loc[1]] == 1: # A is crashing into a car
                self.crash = True
                self.road[loc[0]][loc[1]] = 0 # Erase current marker
                self.road[loc[0]][loc[1]] = -1 # Add marker for crash
                self.cars[0] == (-1, -1)
        elif action == 'S':
            self.done = False
        else:
            raise ValueError(f"Incorrect action specified: {action}")

    def get_percept(self):
        """
        Description
        -------
        This function returns the percept - a (3 x 3) grid sized partial view of the road environment,
        where AA's current location is at the center of the grid. It has the following markers:
            0: empty cell
            1: obstacle car
            2: autonomous agent (AA)
            -1: edge of the road (left, right and bottom edge of the road)
            10: end of the road (top edge of the road)

        """
        partial_state = [[0 for _ in range(3)] for _ in range(3)]

        AA_loc = self.cars[0]
        for x, r in enumerate(range(AA_loc[0]+1, AA_loc[0]-2, -1)):
            for y, c in enumerate(range(AA_loc[1]-1, AA_loc[1]+2, 1)):
                if r < 0:
                    partial_state[x][y] = -1
                elif r == self.height:
                    partial_state[x][y] = 10
                elif c < 0 or c == self.width:
                    partial_state[x][y] = -1
                else:
                    partial_state[x][y] = self.road[r][c]

        return partial_state

    def step(self, action):
        """
        Description
        -----------
        Execute the action taken by the AA.
        Agents's actions are deterministic. However, environment's stochasticity lies in the uncerternity of location of other cars.

        """
        self.__update_environment()
        self.perform_AA_action(action)
        if self.crash == True:
            print("There is a crash on the road!")
            self.display_environment()
            sys.exit(0)
        elif self.done == True:
            print("AA has successfully navigated the road!")
            sys.exit(0)
        elif self.done == False:
            print("Stopping simulation!")
            sys.exit(0)
        # self.display_environment()
        # self.display_percept()

    def step_RL(self, action):
        """
        Description
        -----------
        Execute the action taken by the AA.
        Agents's actions are deterministic. However, environment's stochasticity lies in the uncerternity of location of other cars.

        """
        self.__update_environment()
        self.perform_AA_action(action)
        if self.crash == True:
            #print("There is a crash on the road!")
            #self.display_environment()
            return -100, True
            sys.exit(0)
        elif self.done == True:
            #print("AA has successfully navigated the road!")
            return 100, True
            sys.exit(0)
        elif self.done == False:
            #print("Stopping simulation!")
            sys.exit(0)
        # self.display_environment()
        return -1, False
        self.display_percept()

    def display_environment(self):
        """
        Description
        -----------
        Display the environment with static view of the cars including the autonomous agent
        Example: Change in autonomous driving agent's location.

        You may use this for debugging only. The AA may not have access to the actual state as displayed in this function.

        """

        display = ""
        for r in range(self.height-1, -1, -1):
            # Display borders
            display += chr(43)
            for c in range(self.width):
                display += chr(45) + chr(45) + chr(45) + chr(43)
            display += "\n"

            # Display road
            display += chr(124)
            for c in range(self.width):
                if self.road[r][c] == 1:
                    # ASCII 94 = ^
                    display += chr(32) + chr(94) + chr(32) + chr(124)
                elif self.road[r][c] == 2:
                    # ASCII 65 = A
                    display += chr(32) + chr(65) + chr(32) + chr(124)
                elif self.road[r][c] == -1:
                    # ASCII 42 = *
                    display += chr(32) + chr(42) + chr(32) + chr(124)
                else:
                    # ASCII 32 = \space
                    display += chr(32) + chr(32) + chr(32) + chr(124)
            display += "\n"

        # Display border
        display += chr(43)
        for c in range(self.width):
            display += chr(45) + chr(45) + chr(45) + chr(43)
        display += "\n"

        print(display)
        return

    def display_percept(self):
        """
        Description
        -----------
        Display the (3 x 3) grid sized percept of the AA based on its current location.

        """
        percept = self.get_percept()

        height, width = len(percept), len(percept[0])
        display = ""
        for r in range(height):
            # Display borders
            display += chr(43)
            for c in range(width):
                display += chr(45) + chr(45) + chr(45) + chr(43)
            display += "\n"

            # Display road
            display += chr(124)
            for c in range(width):
                if percept[r][c] == 1:
                    # ASCII 94 = ^
                    display += chr(32) + chr(94) + chr(32) + chr(124)
                elif percept[r][c] == 2:
                    # ASCII 65 = A
                    display += chr(32) + chr(65) + chr(32) + chr(124)
                elif percept[r][c] == -1:
                    # ASCII 120 = x
                    display += chr(32) + chr(120) + chr(32) + chr(124)
                elif percept[r][c] == 10:
                    # ASCII 71 = G
                    display += chr(32) + chr(71) + chr(32) + chr(124)
                else:
                    # ASCII 32 = \space
                    display += chr(32) + chr(32) + chr(32) + chr(124)
            display += "\n"

        # Display border
        display += chr(43)
        for c in range(width):
            display += chr(45) + chr(45) + chr(45) + chr(43)
        display += "\n"

        print(display)
        return

## State

In [ ]:
# state.py

import copy

"""
A state must define methods such as get_percept, generate_successor, get_legal_actions, etc. as required
depending on the type of agent for which an interface is being built.
States are used by the AutonomousDriving object to capture the actual state of the environment/road and
it can be used by agents to reason about driving.
Every agent type has a different state information available to it.
This is a super class for all types of state information.

"""

class ManualState():
    """
    The full state or percept is displayed to a user engaging in manual control.
    You may modify this at your will but it is unnecessary.

    """

    def __init__(self, env):
        self.__env = env

    def display_state(self):
        # self.__env.display_environment() # Comment this if you do not wish to display the road at every step
        self.__env.display_percept() # Comment this if you do not wish to display the percept at every step
        pass

class RandomState():
    """
    The full state or percept is displayed when random agent control is chosen.
    You may modify this at your will but it is unnecessary.

    """

    def __init__(self, env):
        self.__env = env

    def display_state(self):
        # self.__env.display_environment() # Comment this if you do not wish to display the road at every step
        self.__env.display_percept() # Comment this if you do not wish to display the percept at every step
        print(self.__env.get_percept())
        pass

class LearningState():
    """
    A learning agent chooses an action at each choice point based on the Q values approximated.
    In this project your learning agent is essentiually an ApproximateQLearningAgent
    Helper functions are defined to access the attributes of the environment.
    We strongly recommend that you use the following functions to write your
    agent function and not access the environment variable directly.

    """

    def __init__(self, env):
        self.__env = env

    def is_terminal(self):
        """
        Description
        -------
        Returns a boolean indicating if there is a crash on the road or
        if the agent has successfully navigated to the end of the road.

        """
        if self.__env.done or self.__env.crash:
            return True
        else:
            return False

    def step(self, action):
        """
        Description
        -------
        Steps through the environment by having the AA perform the given action.

        """
        next_state = copy.deepcopy(LearningState(self.__env))
        reward, result = next_state.__env.step_RL(action)

        return next_state, reward, result

    def get_road_data(self):
        """
        Description
        -------
        Returns access to the road data with the following markers.
            0: empty cell
            1: obstacle car
            2: autonomous agent (AA)
           -1: crash

        """
        if self.is_terminal():
            return None
        else:
            return self.__env.road

    def get_car_locations(self):
        """
        Description
        -------
        Returns access to all the car locations.
        Location of a car is given by (r, c)
        where r is the index of rows and c is the index of columns in the road array.
        The AA's location is stored at index 0.
        If a car is no longer on the road, its location is None.


        """
        if self.is_terminal():
            return None
        else:
            return self.__env.cars

    def get_height(self):
        """
        Description
        -------
        Returns access to the height of the road.

        """
        return self.__env.height

    def get_width(self):
        """
        Description
        -------
        Returns access to the width of the road.

        """
        return self.__env.width


## Agent

In [ ]:
import random
import numpy as np

class Agent:
    """
    An agent must define a get_action method, but may also define
    other methods which will be called if they exist.
    This is a super class for any agent type.

    """

    def __init__(self):
        """
        Description
        -----------
        The list of available actions for all cars are:
        Forward - 'F'
        Left - 'L'
        Right - 'R'
        Wait - 'W'

        """
        self.available_actions = ['F', 'L', 'R', 'W']

    def get_action(self, state):
        """
        The Agent will receive a State of the environment (based on the agent type) and
        must return an action from the available actions {Forward - 'F', Left - 'L', Right - 'R' and Wait - 'W'}.
        """
        pass

#### helper functions

In [ ]:
def normalize(value, min_value, max_value):
    """
    Description
    ----------
    Normalizes a given value between 0-1

    """
    return (value - min_value) / (max_value - min_value)

def manhattan_distance(loc1, loc2):
    """
    Description
    ----------
    Returns the Manhattan distance between points loc1 and loc2

    """
    return abs( loc1[0] - loc2[0] ) + abs( loc1[1] - loc2[1] )

### Manual Agent

In [ ]:
class ManualAgent(Agent):
    """
    A manual agent is used to control the Autonomous Agent ('A') manually by the user.

    """

    def get_action(self, percept):
        "*** YOUR CODE HERE ***"
        print("Enter Action (Forward - 'F', Left - 'L', Right - 'R', Wait - 'W', Stop - 'S'):\n")
        action = input()
        return action

### Random Agent

In [ ]:
class RandomAgent(Agent):
    """
    A random agent chooses an action randomly at each choice point from the list of available actions.

    """

    def get_action(self, percept):
        "*** YOUR CODE HERE ***"
        action = random.choice(self.available_actions) #should choose an action among the legal actions available
        print(f"Random action chosen: {action}")
        input("Press enter to step through.")
        return action


### Learning Agent

In [ ]:
class LearningAgent(Agent):
    """
    A learning agent chooses an action at each choice point based on the Q values approximated.
    In this project your learning agent is essentiually an ApproximateQLearningAgent

    """

    def __init__(self, num_features=1, custom_weights=False, weights=None, alpha=0.1, gamma=0.99):
        self.alpha = alpha
        self.gamma = gamma
        self.num_features = num_features
        self.decay_rate = 0.99
        self.epsilon = 1

        if custom_weights:
            self.weights = np.array(weights, dtype=float)
        else:
            self.weights = np.random.rand(num_features)

        super().__init__()

    def get_weights(self):
        return self.weights

    def get_features(self, state, action):
        """
        Description
        ----------
        This function returns a vector of features f for the given state action pair

        Compute: f = [f_1(s, a), f_2(s, a), ... , f_n(s, a)]

        """
        "*** YOUR CODE HERE ***"
        feats = np.zeros(self.num_features)
        road = state.get_road_data()
        if road is None:
            return feats

        car_locs = state.get_car_locations()
        aa = car_locs[0]
        h, w = state.get_height(), state.get_width()

        dr, dc = {"F": (1, 0), "L": (0, -1), "R": (0, 1), "W": (0, 0)}[action]
        nxt = (aa[0] + dr, aa[1] + dc)
        at_exit = nxt[0] >= h
        in_bounds = 0 <= nxt[0] < h and 0 <= nxt[1] < w

        if at_exit:
            feats[0] = 1.0
        elif in_bounds:
            feats[0] = nxt[0] / h

        if self.num_features > 1 and (at_exit or (in_bounds and road[nxt[0]][nxt[1]] != 1)):
            feats[1] = 1.0

        if self.num_features > 2:
            if at_exit:
                feats[2] = 1.0
            elif in_bounds:
                others = (manhattan_distance(nxt, loc) for idx, loc in car_locs.items() if idx != 0 and loc is not None)
                feats[2] = min(min(others, default=float("inf")), MAX_VIEW) / MAX_VIEW

        return feats

    def get_Q_value(self, state, action):
        """
        Description
        ----------
        This function returns the Q value; Q(state, action) = weightVector . featureVector
        where . is the dotProduct operator

        Compute: Q(s, a) = w . f = w_1 * f_1 + w_2 * f_2 + ... + w_n * f_n

        """
        "*** YOUR CODE HERE ***"
        return float(np.dot(self.weights, self.get_features(state, action)))

    def update(self, state, action, next_state, reward):
        """
        Description
        ----------
        This function updates the weights based on the given transition

        Compute:

        w_{t+1} = w_t + alpha * delta_{t} * f
        delta_{t} = r + gamma * max_a' Q(s', a') - Q(s,a))

        """
        "*** YOUR CODE HERE ***"
        feats = self.get_features(state, action)
        delta = reward + self.gamma * self.compute_max_Q_value(next_state) - float(np.dot(self.weights, feats))
        self.weights += self.alpha * delta * feats

    def compute_max_Q_value(self, state):
        """
        Description
        ----------
        This function returns the max over all Q(state, action)
        for all legal/available actions for the given state
        Note that if there are no legal actions, which is the case at the
        terminal state, you should return a value of 0.0.

        Compute: max_a' Q(s', a')

        """
        if (state.is_terminal()):
            return 0.0
        max_value = float('-inf')
        for action in self.available_actions:
            value = self.get_Q_value(state, action)
            if value > max_value:
                max_value = value
        return max_value

    def get_action(self, state):
        """
        Description
        ----------
        This function returns the best action based on the self.weights it has learned.

        """
        max_qvalue = float('-inf')
        best_action = None
        for action in self.available_actions:
            qvalue = self.get_Q_value(state, action)
            if qvalue > max_qvalue:
                max_qvalue = qvalue
                best_action = action
        return best_action

    def train(self, state):
        """
        Description
        ----------
        This function learns the weights for your approximate Q learning agent
        by training for a single episode given the initialization of the road.

        """

        cum_reward = 0
        done=False
        while not (done):
            # Choose an action epsilon greedily
            if np.random.rand() < self.epsilon:
                action = np.random.choice(self.available_actions)
            else:
                action = self.get_action(state)

            # Take the action and observe the next state and reward
            next_state, reward, done = state.step(action)
            # Update weights by approximating Q value
            self.update(state, action, next_state, reward)

            # Move to the next state
            state = next_state
            cum_reward += reward

        return cum_reward


## Problem

In [ ]:
# from environment import Environment
# from agent import *
# from state import *

import copy
import random


class AutonomousDriving:
    """
    This class creates a problem instance by initializing a specific agent type and simulates the autonomous driving environment.

    """
    def __init__(self, agent_type, height, width, occupancy, init, seed, episodes, features, custom_weights=False, weights=None):
        self.agent = None
        self.env = Environment(height, width, occupancy, init, seed)

        if agent_type == 'manual':
            self.agent = ManualAgent()
        elif agent_type == 'random':
            self.agent = RandomAgent()
        elif agent_type == 'learning':
            self.agent = LearningAgent(features, custom_weights, weights)
        else:
            raise ValueError(f"Incorrect agent specified: {agent}")

    def run(self, agent_type, episodes=50000):
        """
        Description
        -----------
        This function calls the simulation function based on the agent initialized.
        and passes it to the step function.
        Thus, simulating the autonomous driving.

        """

        if agent_type == 'manual':
            self.run_manual_control()
        elif agent_type == 'random':
            self.run_random_control()
        elif agent_type == 'learning':
            self.run_learning_control(episodes)

    def run_manual_control(self):
        self.env.display_environment()
        while True:
            state_obj = ManualState(copy.deepcopy(self.env))
            state_obj.display_state()
            action = self.agent.get_action(None)
            self.env.step(action)

    def run_random_control(self):
        self.env.display_environment()
        while True:
            state_obj = RandomState(copy.deepcopy(self.env))
            state_obj.display_state()
            action = self.agent.get_action(None)
            self.env.step(action)

    def run_learning_control(self, num_episodes):
        W = self.train(num_episodes)
        self.test()

    def train(self, num_episodes):
        self.env.display_environment()
        total_reward = 0
        # Training
        print("training...")
        for eps in range(num_episodes):
            seed = random.randrange(100)
            #Create a new environment configuration
            env = Environment(self.env.height, self.env.width, self.env.occupancy, self.env.init, seed)
            obj = LearningState(env)
            total_reward += self.agent.train(obj)
            if(eps%1000==0):
                print(f"Episode {eps + 1}, Total Reward: {total_reward}")
                self.agent.epsilon = self.agent.epsilon * self.agent.decay_rate

        return self.agent.get_weights()

    def test(self):
        print("testing...")
        while True:
            state_obj = LearningState(copy.deepcopy(self.env))
            action = self.agent.get_action(state_obj)
            print(action)
            self.env.step(action)


## Playground

#### Feel free to play around with different environment configurations by changing the following parameters.
```
height - Road height (int), default: 10
width - Road width (int), default: 4
occupancy - Occupancy of the road with other cars. Enter a value between 0 and 0.3, default: 0.1
init - Initial location of 'A' w.r.t. the height of the road. \nENter a value between 0 and 0.9, default: 0.2
seed - To initialize cars on the road (int), default: 3
episodes - Number of training episodes (int) - default: 50000"
features - Number of features that will be extracted (int) - default: 1
```

In [ ]:
height = 10
width = 4
occupancy = 0.1
init = 0.2
seed = 3
episodes = 50000
num_features = 1

### Playground - Manual Agent

In [ ]:
# agent = 'manual'

# driver = AutonomousDriving(
#         agent_type=agent,
#         height=height,
#         width=width,
#         occupancy=occupancy,
#         init=init,
#         seed=seed
#     )

# driver.run(agent)

### Playground - Random Agent

In [ ]:
# agent = 'random'

# driver = AutonomousDriving(
#         agent_type=agent,
#         height=height,
#         width=width,
#         occupancy=occupancy,
#         init=init,
#         seed=seed
#     )

# driver.run(agent)

### Playground - Learning Agent

In [ ]:
""" Uncomment the following code to run the learning agent in Playground mode """
# agent = 'learning'

# driver = AutonomousDriving(
#         agent_type=agent,
#         height=height,
#         width=width,
#         occupancy=occupancy,
#         init=init,
#         seed=seed,
#         episodes=episodes,
#         features=num_features
#     )

# driver.run(agent, episodes)

+---+---+---+---+
|   | ^ |   |   |
+---+---+---+---+
|   |   | ^ |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
|   |   |   | ^ |
+---+---+---+---+
| A |   |   |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+
|   |   |   |   |
+---+---+---+---+

training...


Exception: Function not defined

## Evaluation

In [ ]:
import io
from contextlib import redirect_stdout

SUCCESS_MSG = 'AA has successfully navigated the road!'
FAILURE_MSG = 'There is a crash on the road!'


def _capture_run(fn, verbose=False):
    """Run a callable, capture notebook output, and swallow SystemExit used by env.step()."""
    buffer = io.StringIO()
    try:
        with redirect_stdout(buffer):
            fn()
    except SystemExit:
        pass
    output = buffer.getvalue()
    if verbose:
        print(output)
    return output

def _scaled_score(raw_score, max_cases=7, max_points=10):
    scaled = round((raw_score / max_cases) * max_points, 1)
    return min(scaled, max_points)

def _run_case(agent_type, case, verbose=False):
    driver = AutonomousDriving(
        agent_type=agent_type,
        height=case['height'],
        width=case['width'],
        occupancy=case['occupancy'],
        init=case['init'],
        seed=case['seed'],
        episodes=0,
        features=1,
    )
    return _capture_run(lambda: driver.run(agent_type), verbose=verbose)

def grade_q3(tests, verbose=False, features=1):
    print('----- Testing Q3 -----')
    score_q3 = 0
    num_test_cases = len(tests['q3']['test'])

    train_case = tests['q3']['train'][str(0)]
    episodes = train_case['episodes']

    driver = AutonomousDriving(
        agent_type='learning',
        height=train_case['height'],
        width=train_case['width'],
        occupancy=train_case['occupancy'],
        init=train_case['init'],
        seed=train_case['seed'],
        episodes=episodes,
        features=int(features),
    )

    learned_weights = driver.train(episodes).tolist()

    for test_case in range(num_test_cases):
        case = tests['q3']['test'][str(test_case)]
        test_driver = AutonomousDriving(
            agent_type='learning',
            height=case['height'],
            width=case['width'],
            occupancy=case['occupancy'],
            init=case['init'],
            seed=case['seed'],
            episodes=episodes,
            features=int(features),
            custom_weights=True,
            weights=learned_weights,
        )

        out = _capture_run(lambda: test_driver.test(), verbose=verbose)

        if SUCCESS_MSG in out:
            score_q3 += 1
            print(f'Testcase {test_case}: PASS')
        elif FAILURE_MSG in out:
            print(f'Testcase {test_case}: FAIL')
        else:
            print(f'Testcase {test_case}: FAIL')
            print('There was an error in running test case: ', test_case)

    return _scaled_score(score_q3)


#### Test Cases

In [ ]:

tests = {
"q3": {
    "train": {
        "0":  {
        "height": 30,
        "width": 5,
        "occupancy": 0.2,
        "init": 0.2,
        "seed": 10,
        "episodes": 50000
            }
        },
    "test": {
        "0":  {
            "height": 30,
            "width": 4,
            "occupancy": 0.1,
            "init": 0.2,
            "seed": 33
            },
        "1": {
            "height": 20,
            "width": 2,
            "occupancy": 0.3,
            "init": 0.2,
            "seed": 5
            },
        "2": {
            "height": 10,
            "width": 3,
            "occupancy": 0.3,
            "init": 0.5,
            "seed": 6
            },
        "3": {
            "height": 30,
            "width": 5,
            "occupancy": 0.1,
            "init": 0.2,
            "seed": 13
            },
        "4": {
            "height": 20,
            "width": 2,
            "occupancy": 0.2,
            "init": 0.4,
            "seed": 10
            },
        "5": {
            "height": 10,
            "width": 3,
            "occupancy": 0.3,
            "init": 0.5,
            "seed": 11
            },
        "6": {
            "height": 30,
            "width": 5,
            "occupancy": 0.2,
            "init": 0.5,
            "seed": 16
            },
        "7": {
            "height": 10,
            "width": 5,
            "occupancy": 0.1,
            "init": 0.1,
            "seed": 13
            },
        "8": {
            "height": 30,
            "width": 5,
            "occupancy": 0.15,
            "init": 0.5,
            "seed": 31
            },
        "9": {
            "height": 2,
            "width": 5,
            "occupancy": 0.3,
            "init": 0.4,
            "seed": 6
            },
        "10": {
            "height": 10,
            "width": 3,
            "occupancy": 0.2,
            "init": 0.5,
            "seed": 10
            }
        }

    }
}


### Grade Q3

In [ ]:
num_features = 3
q3_results = grade_q3(tests, verbose=False, features=num_features)

print('===========================')
print(f'Q3 score:\t{q3_results} / 10.0')
print('===========================')
